# Step 0: Import requisite libraries

In [200]:
import math
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn import preprocessing, ensemble, tree
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import matplotlib as plt
import seaborn

# Step 1: Import and cleaning of data

In [201]:
df = pd.read_csv("Baza_customer_Telecom_v2.csv")
df

,PID,CRM_PID_Value_Segment,EffectiveSegment,Billing_ZIP,KA_name,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,CHURN
0,123759242,Bronze,SOHO,6000.0,VM,2,NaN,NaN,2,40.17,0.0,40.17,NaN,No
1,126145737,Bronze,SOHO,6400.0,VM,3,NaN,NaN,3,40.17,0.0,40.17,13.39,No
2,123506355,Bronze,SOHO,6000.0,DI,2,3.0,NaN,5,40.17,0.0,40.17,20.09,No
3,112595585,Bronze,SOHO,4400.0,MT,1,2.0,NaN,3,40.17,0.0,40.17,40.17,No
4,115097935,Iron,SOHO,4000.0,AD,2,1.0,NaN,3,40.17,0.0,40.17,20.09,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8448,115901902,Iron,SOHO,4000.0,VU,2,1.0,NaN,3,56.50,0.0,56.50,28.25,No
8449,115800273,Bronze,SOHO,4006.0,VU,5,NaN,NaN,5,56.50,0.0,56.50,11.30,No
8450,160049889,Bronze,SOHO,4003.0,VU,3,1.0,NaN,4,56.50,0.0,56.50,18.83,No
8451,160095203,Bronze,SOHO,4023.0,VU,3,NaN,NaN,3,56.50,0.0,56.50,18.83,Yes


In [202]:
print(f"Length of dataset before cleaning: {len(df)}")
# Cleaning for PID
df = df.loc[df["PID"].notnull() & df["PID"].str.fullmatch(r"[0-9]*(SMB)?")] # Some PIDs were scientific notation (invalid); valid format is integer optionally followed by "SMB"

# Cleaning for value segment
df = df.loc[df["CRM_PID_Value_Segment"].notnull()]
df["CRM_PID_Value_Segment"] = df["CRM_PID_Value_Segment"].replace("Sliver", "Silver") # Fix assumed typo in one entry

# Cleaning for effective segment
df = df.loc[df["EffectiveSegment"].notnull()]

# Cleaning for ZIP
df = df.loc[df["Billing_ZIP"] % 1 == 0] # Remove non-integer ZIP codes (invalid; includes nulls)

# Cleaning for KA_name
df = df.loc[df["KA_name"].notnull()]
df["KA_name"] = df["KA_name"].replace("AD?", "AD") # Grouped AD? with AD under assumption that unsure attribution is correct

# Cleaning for active_subscribers
df = df.loc[df["Active_subscribers"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for not_active_subscribers
df["Not_Active_subscribers"] = df["Not_Active_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none inactive; minimum non-NaN value is 1)

# Cleaning for suspended_subscribers
df["Suspended_subscribers"] = df["Suspended_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none suspended; minimum non-NaN value is 1)

# Cleaning for total_SUBs
df = df.loc[df["Total_SUBs"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for AvgMobileRevenue
df = df.loc[df["AvgMobileRevenue "].notnull()] # Redundant (no nulls) but included for consistency
df = df.rename(columns={"AvgMobileRevenue ": "AvgMobileRevenue"}) # Remove trailing space from column name

# Cleaning for AvgFIXRevenue
df = df.loc[df["AvgFIXRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for TotalRevenue
df = df.loc[df["TotalRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for ARPU
df = df.loc[df["ARPU"].notnull()]

# Cleaning for CHURN
df = df.loc[df["CHURN"].isin(["No", "Yes"])] # Remove invalid CHURN values (only valid values are "No" and "Yes"); redundant (no invalids) but included for consistency

print(f"Length of dataset after cleaning: {len(df)}") # Small length change, hence valid to discard nulls/invalid entries

# Source - https://stackoverflow.com/a/48035423
# Posted by BENY, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-02, License - CC BY-SA 4.0

df.apply(lambda x : pd.factorize(x)[0]).corr(method='pearson', min_periods=1)
df.groupby("CRM_PID_Value_Segment")["TotalRevenue"].mean().sort_values()

Length of dataset before cleaning: 8453
Length of dataset after cleaning: 8419


CRM_PID_Value_Segment
Iron         59.888252
Bronze       79.287700
Silver      146.561462
SE          166.550278
Lead        170.716667
Gold        239.062784
SME         308.828669
Platinum    360.134227
Name: TotalRevenue, dtype: float64

# Step 2: Data Preprocessing

In [203]:
X = df.drop(columns=["CHURN"])
y = df["CHURN"].map({"No": 0, "Yes": 1}) # Convert CHURN to binary values (0 for No, 1 for Yes)

X["PID_is_SMB"] = X["PID"].str.endswith("SMB").astype("string") # Create binary feature indicating if PID ends with "SMB"
X["PID"]=X["PID"].str.replace("SMB", "").astype(int) # Remove "SMB" from PID and convert to integer

# Definition of categorical and numerical features for preprocessing
cat_features = ["CRM_PID_Value_Segment", "EffectiveSegment", "KA_name", "PID_is_SMB"]
num_features = ["Billing_ZIP", "Active_subscribers", "Not_Active_subscribers", "Suspended_subscribers", "Total_SUBs", "AvgMobileRevenue", "AvgFIXRevenue", "TotalRevenue", "ARPU"]

# Selection of features for model training

train_features = cat_features + num_features

class_weights = {0: 1, 1: 15.397085610200364}

# train_features = [
#     "PID_is_SMB",
#     "CRM_PID_Value_Segment", 
#     "Active_subscribers",
#     "AvgMobileRevenue",
#     "TotalRevenue"
#     ]

X = X[train_features]

# Definition of ColumnTransformer for preprocessing
# c_transformer = ColumnTransformer(
#     transformers=[
#         ("categorical", preprocessing.OneHotEncoder(handle_unknown="ignore"), list(set(cat_features) & set(train_features)) ),
#         ("numerical", "passthrough", list(set(num_features) & set(train_features)) )
#     ]
# )

# print(f"Column transformer has categorical features {list(set(cat_features) & set(train_features))} and numerical features {list(set(num_features) & set(train_features))}")


# XGBoost prep
for col in cat_features:
    X[col] = X[col].astype('category')

# Model pipeline
clf_p = Pipeline([
    # ("preprocessor", c_transformer),
    # ("classifier", ensemble.RandomForestClassifier(
    #     n_estimators=500,
    #     class_weight="balanced",
    #     n_jobs=-1,
    #     random_state=42))

    # ("classifier", tree.DecisionTreeClassifier(
    #     class_weight="balanced",
    #     splitter="best",
    #     max_depth=None,
    #     random_state=42))

    ("classifier", xgb.XGBClassifier(
        enable_categorical = True,
        tree_method="hist",
        n_estimators = 1000,
        learning_rate = 0.1,
        max_depth = None,
        objective = 'binary:logistic',
        eval_metric = 'aucpr',
        scale_pos_weight = sum(class_weights.values()) / sum(class_weights.keys()),
        max_delta_step = 1,
        min_child_weight = 5,
        random_state = 42
    ))
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit model
clf_p.fit(X_train, y_train)

# Predict on test set
y_pred_p = clf_p.predict_proba(X_test)
t_df = pd.DataFrame(X_test)
t_df["Predicted_Churn_Probability"] = y_pred_p[:, 1] # Get probabilities for positive class (CHURN = Yes)
t_df["Predicted_Churn"] = (t_df["Predicted_Churn_Probability"] >= 0.5).astype(int) # Convert probabilities to binary predictions using threshold of 0.5
t_df["Actual_Churn"] = y_test.values # Add actual CHURN values for comparison
t_df["Prediction_type"] = np.select(
    [
        (t_df["Predicted_Churn"] == 1) & (t_df["Actual_Churn"] == 1),
        (t_df["Predicted_Churn"] == 0) & (t_df["Actual_Churn"] == 0),
        (t_df["Predicted_Churn"] == 1) & (t_df["Actual_Churn"] == 0),
        (t_df["Predicted_Churn"] == 0) & (t_df["Actual_Churn"] == 1)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

# Evaluate model performance
y_pred = y_pred_p[:, 1] >= 0.4
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
t_df.to_csv("Predicted_Churn_Results.csv", index=False) # Save predictions to CSV for further analysis
t_df[t_df["Prediction_type"] == "True Positive"].sort_values(by="Predicted_Churn_Probability", ascending=False).head(20) # Sort by predicted probability for easier analysis


              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1575
           1       0.11      0.11      0.11       109

    accuracy                           0.88      1684
   macro avg       0.52      0.52      0.52      1684
weighted avg       0.88      0.88      0.88      1684

[[1478   97]
 [  97   12]]


,CRM_PID_Value_Segment,EffectiveSegment,KA_name,PID_is_SMB,Billing_ZIP,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,Predicted_Churn_Probability,Predicted_Churn,Actual_Churn,Prediction_type
3436,Platinum,SOHO,DI,False,6000.0,1,20.0,0.0,21,181.17,0.0,181.17,181.17,0.972945,1,1,True Positive
7679,SME,SME,Tatiana Trifonova,False,4400.0,53,6.0,0.0,59,419.83,0.0,419.83,7.92,0.834238,1,1,True Positive
6230,Bronze,SOHO,Tatiana Trifonova,False,4850.0,4,0.0,0.0,4,106.00,0.0,106.00,26.50,0.806425,1,1,True Positive
1951,Bronze,SOHO,AD,False,8000.0,9,0.0,0.0,9,105.67,0.0,105.67,11.74,0.721867,1,1,True Positive
7628,SME,SME,VM,False,6260.0,13,1.0,0.0,14,307.00,0.0,307.00,23.62,0.632120,1,1,True Positive
4155,Gold,VSE,AD,False,6400.0,13,2.0,0.0,15,244.17,0.0,244.17,18.78,0.590417,1,1,True Positive
3328,Bronze,SOHO,VU,False,4600.0,6,0.0,0.0,6,174.83,0.0,174.83,29.14,0.572327,1,1,True Positive
2042,Bronze,SOHO,RJ,False,6600.0,5,2.0,0.0,7,109.17,0.0,109.17,21.83,0.520610,1,1,True Positive


In [204]:
t_df[t_df["Prediction_type"] == "False Positive"].sort_values(by="Predicted_Churn_Probability", ascending=False).head(20) # Sort by predicted probability for easier analysis

,CRM_PID_Value_Segment,EffectiveSegment,KA_name,PID_is_SMB,Billing_ZIP,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,Predicted_Churn_Probability,Predicted_Churn,Actual_Churn,Prediction_type
7547,Gold,VSE,Ginka Vachkova,False,4021.0,22,6.0,1.0,29,490.17,0.0,490.17,22.28,0.997984,1,0,False Positive
7520,SME,SME,DI,False,6000.0,7,2.0,0.0,9,188.17,0.0,188.17,26.88,0.993554,1,0,False Positive
7184,Bronze,SOHO,Daniela Stefanova,False,4000.0,2,0.0,0.0,2,63.67,0.0,63.67,31.84,0.983624,1,0,False Positive
2874,Silver,SOHO,Tatiana Trifonova,False,4000.0,20,0.0,1.0,21,147.83,0.0,147.83,7.39,0.976746,1,0,False Positive
7696,SME,SME,Tatiana Trifonova,False,4150.0,20,1.0,0.0,21,144.17,0.0,144.17,7.21,0.975204,1,0,False Positive
270,Bronze,SOHO,VM,False,6000.0,4,0.0,0.0,4,50.17,0.0,50.17,12.54,0.951999,1,0,False Positive
4709,Gold,VSE,Daniela Stefanova,False,4000.0,15,0.0,0.0,15,325.33,0.0,325.33,21.69,0.937914,1,0,False Positive
3546,Silver,SOHO,AD,False,6000.0,10,3.0,0.0,13,188.17,0.0,188.17,18.82,0.937087,1,0,False Positive
3165,Silver,VSE,Tatiana Trifonova,False,4230.0,15,0.0,0.0,15,165.17,0.0,165.17,11.01,0.926119,1,0,False Positive
3614,Gold,SOHO,RJ,False,6300.0,5,0.0,0.0,5,193.17,0.0,193.17,38.63,0.891396,1,0,False Positive


In [205]:
# Testing per-group
test_segment = "Bronze"
segment_test = y_test[X_test["CRM_PID_Value_Segment"] == test_segment]
segment_pred = y_pred[X_test["CRM_PID_Value_Segment"] == test_segment]

print(classification_report(segment_test, segment_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95       703
           1       0.20      0.14      0.17        49

    accuracy                           0.91       752
   macro avg       0.57      0.55      0.56       752
weighted avg       0.89      0.91      0.90       752

